# 09. Probabilistic Forecasting & Conformal Prediction

## Background

A point forecast answers 'what will sales be next Tuesday?' Probabilistic forecasting answers the more useful question: 'what range of values should I plan for?' — with calibrated uncertainty. A 90% prediction interval that contains the true value only 60% of the time is misleading and causes poor inventory decisions.

Three main approaches to probabilistic forecasting:

1. **Distributional**: assume a parametric distribution (Normal, Negative Binomial),    predict its parameters. Fast, interpretable, but wrong when the assumed distribution    is wrong.

2. **Quantile regression**: directly predict quantiles via pinball loss. No distributional    assumption, but requires separate training per quantile and can produce non-monotonic    quantile crossings.

3. **Conformal prediction**: distribution-free, coverage-guaranteed intervals.    Calibrate on a held-out dataset to achieve exact frequentist coverage guarantees.    Works on top of any point forecaster.

## What You'll Learn

- Pinball loss for quantile regression
- Conformal prediction intervals for time series (EnbPI)
- Calibration evaluation: MACE (Mean Absolute Coverage Error)
- Marginal vs conditional coverage: what conformal prediction guarantees

## Why This Matters

Every production forecasting system needs calibrated uncertainty. Safety stock decisions, capacity planning, financial risk management all depend on knowing not just the forecast but the reliable range. Miscalibrated intervals waste inventory or cause stockouts. Conformal prediction provides mathematically guaranteed coverage — a promise no other approach makes without assumptions.


## Quantile Regression Forecaster

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

QUANTILES = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]

def pinball_loss(y_pred: torch.Tensor, y_true: torch.Tensor,
                  quantile: float) -> torch.Tensor:
    errors = y_true - y_pred
    return torch.max(quantile * errors, (quantile - 1) * errors).mean()

class QuantileForecaster(nn.Module):
    def __init__(self, input_size: int, horizon: int,
                 quantiles: List[float], hidden: int = 256):
        super().__init__()
        self.quantiles = quantiles
        self.horizon = horizon
        self.backbone = nn.Sequential(
            nn.Linear(input_size, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        # One output head per quantile
        self.heads = nn.ModuleList([
            nn.Linear(hidden, horizon) for _ in quantiles
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.backbone(x)
        return torch.stack([head(h) for head in self.heads], dim=-1)  # (B, H, Q)

    def multi_quantile_loss(self, preds: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # preds: (B, H, Q), targets: (B, H)
        losses = []
        for i, q in enumerate(self.quantiles):
            losses.append(pinball_loss(preds[:, :, i], targets, q))
        return torch.stack(losses).mean()

# Generate data
def make_dataset(n=1000):
    t = np.arange(n)
    y = 50 + 0.1*t + 20*np.sin(2*np.pi*t/52) + np.random.laplace(0, 5, n)
    return y

series = make_dataset(1000)
CONTEXT, HORIZON = 52, 12

def make_windows(series, context, horizon):
    X, y = [], []
    for i in range(len(series) - context - horizon):
        X.append(series[i:i+context])
        y.append(series[i+context:i+context+horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = make_windows(series, CONTEXT, HORIZON)
mean, std = X[:800].mean(), X[:800].std()
X_norm = (X - mean) / std
y_norm = (y - mean) / std

X_train, y_train = torch.tensor(X_norm[:700]), torch.tensor(y_norm[:700])
X_cal, y_cal = torch.tensor(X_norm[700:850]), torch.tensor(y_norm[700:850])
X_test, y_test = torch.tensor(X_norm[850:]), torch.tensor(y_norm[850:])

model = QuantileForecaster(CONTEXT, HORIZON, QUANTILES)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print('Training quantile forecaster...')
for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train)
    loss = model.multi_quantile_loss(preds, y_train)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 25 == 0:
        print(f'  Epoch {epoch+1}: loss={loss.item():.4f}')


## Calibration Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test).numpy() * std + mean
y_test_np = y_test.numpy() * std + mean

# Map quantile index: 0.05, 0.10, ..., 0.95
Q_IDX = {q: i for i, q in enumerate(QUANTILES)}

print('Calibration Report:')
print(f'{"Quantile":>10} {"Expected Cov":>14} {"Actual Cov":>12} {"Error":>8}')
print('-'*48)

coverage_errors = []
for q in QUANTILES:
    # For quantile q: fraction of actuals below q-th predicted quantile
    q_preds = test_preds[:, :, Q_IDX[q]]  # (N, H)
    below = (y_test_np <= q_preds).mean()
    error = abs(below - q)
    coverage_errors.append(error)
    print(f'{q:>10.2f} {q:>14.2f} {below:>12.4f} {error:>8.4f}')

mace = np.mean(coverage_errors)
print(f'\nMACE (Mean Abs Coverage Error): {mace:.4f}')
print(f'Interpretation: on average, quantiles are off by {mace:.2%}')

# PI coverage for key intervals
print(f'\nPrediction Interval Coverage:')
for lo, hi, target in [(0.05, 0.95, 0.90), (0.10, 0.90, 0.80), (0.25, 0.75, 0.50)]:
    lo_preds = test_preds[:, :, Q_IDX[lo]]
    hi_preds = test_preds[:, :, Q_IDX[hi]]
    cov = np.mean((y_test_np >= lo_preds) & (y_test_np <= hi_preds))
    print(f'  [{lo},{hi}] -> {target:.0%} PI: {cov:.2%} (error={abs(cov-target):.2%})')


## Conformal Prediction Intervals

In [ ]:
def compute_conformal_intervals(model, X_cal, y_cal, X_test,
                                  coverage: float = 0.90) -> Tuple[np.ndarray, np.ndarray]:
    '''EnbPI-style conformal prediction for time series.

    1. On calibration set: compute residuals = |actual - median_forecast|
    2. Find (1-alpha) quantile of these residuals
    3. Test interval = [median - q_hat, median + q_hat]

    Guarantee: covers true value >= coverage fraction under exchangeability.
    '''
    alpha = 1 - coverage

    model.eval()
    with torch.no_grad():
        cal_preds = model(X_cal).numpy() * std + mean  # (N_cal, H, Q)
        test_preds_raw = model(X_test).numpy() * std + mean

    y_cal_np = y_cal.numpy() * std + mean

    # Median forecast for calibration set
    cal_median = cal_preds[:, :, Q_IDX[0.50]]  # (N_cal, H)

    # Nonconformity scores: absolute residuals
    scores = np.abs(y_cal_np - cal_median).flatten()

    # Find conformal quantile
    n_cal = len(scores)
    q_hat = np.quantile(scores, np.ceil((n_cal + 1) * (1 - alpha)) / n_cal)

    # Test intervals
    test_median = test_preds_raw[:, :, Q_IDX[0.50]]
    lower = test_median - q_hat
    upper = test_median + q_hat

    return lower, upper, q_hat

lower, upper, q_hat = compute_conformal_intervals(model, X_cal, y_cal, X_test, coverage=0.90)

# Evaluate coverage
conformal_coverage = np.mean((y_test_np >= lower) & (y_test_np <= upper))

print(f'Conformal Prediction Intervals (target: 90%)')
print(f'  Calibration set size: {len(X_cal)}')
print(f'  Conformal quantile (q_hat): {q_hat:.4f}')
print(f'  Interval width: {(upper - lower).mean():.4f} (constant by design)')
print(f'  Actual coverage: {conformal_coverage:.2%}')
print(f'  Coverage error: {abs(conformal_coverage - 0.90):.2%}')
print()
print('Guarantee: under exchangeability, coverage >= 1-alpha with high probability')
print('No distributional assumptions required — works for any underlying model.')
